In [19]:
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              mean_squared_error, r2_score, mean_absolute_error)
                              

In [7]:
with open('train_data.pickle', 'rb') as f:
        train_data = pickle.load(f)
        x = train_data['X']
        y = train_data['y']

In [10]:
print(x.shape)
print(y[:10], np.unique(y))


(124, 13)
[0 0 0 0 1 0 2 1 2 0] [0 1 2]


очевидно, это задача классификации

In [12]:
print(np.isnan(x).sum())
print(np.isnan(y).sum())

0
0


данные без пропусков, можем приступать к обучению

In [27]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}
scoring_metrics = ['accuracy', 'f1_weighted']

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

In [22]:
for name, model in models.items():
    print(f"Model name: {name}")
    
    model_results = {}
    
    if name == 'Logistic Regression':
        x_data = x_scaled
    else:
        x_data = x
    
    for metric in scoring_metrics:
        scores = cross_val_score(model, x_data, y, cv=cv, scoring=metric)
        model_results[metric] = {
            'mean': scores.mean(),
            'std': scores.std(),
            'scores': scores
        }
        print(f"{metric.upper():15}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")
    
    results[name] = model_results

Model name: Logistic Regression
ACCURACY       : 0.9840 (+/- 0.0392)
F1_WEIGHTED    : 0.9842 (+/- 0.0388)
Model name: Random Forest
ACCURACY       : 0.9760 (+/- 0.0640)


/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: invalid value encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features

F1_WEIGHTED    : 0.9757 (+/- 0.0652)


In [50]:
best_model_name = None
best_f1_score = -1

for name, model_results in results.items():
    f1_mean = model_results['f1_weighted']['mean']
    if f1_mean > best_f1_score:
        best_f1_score = f1_mean
        best_model_name = name

print(f"\nЛучшая модель по F1-weighted: {best_model_name}")
print(f"F1-weighted = {best_f1_score:.4f}")


Лучшая модель по F1-weighted: Logistic Regression
F1-weighted = 0.9842


In [23]:
for name, model_results in results.items():
    accur_mean = model_results['accuracy']['mean']
    print(f'{name} | mean_acc: {accur_mean}')

Logistic Regression | mean_acc: 0.984
Random Forest | mean_acc: 0.976


In [24]:
base_model = LogisticRegression(max_iter=1000, random_state=42)
final_model = base_model.fit(x_data, y)

/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: overflow encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: invalid value encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/nothottryhard/projects/mlops-vehicle-insurance-claims/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=

In [26]:
with open('model_best.pickle', 'wb') as f:
    pickle.dump(final_model,f)